# Scientific Python
## Central European University

## 06 NetworkX, scipy.stats, scikit-learn -- exercises

Instructor: Márton Pósfai

Email: posfaim@ceu.edu

*Don't forget:* use the Slack channel for discussion, to ask questions, or to show solutions to exercises that are different from the ones provided in the notebook. [Slack channel](http://www.personal.ceu.edu/staff/Marton_Posfai/slack_forward.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import networkx as nx
import seaborn as sns
from scipy import stats, optimize
from sklearn import linear_model, model_selection, metrics
import math
import urllib.request
from bs4 import BeautifulSoup
import json

## Final problem

### Part I

Write a function that returns two lists conataining the authors and the titles of the books on the New York Times best seller list.
* Download and parse the page from here: https://www.nytimes.com/books/best-sellers/combined-print-and-e-book-fiction/
* The book titles are encased in a tag with attribute itemprop="name"
* The authors are in a tag itemprop="author"

In [ ]:
def bestseller():
    url = "https://www.nytimes.com/books/best-sellers/combined-print-and-e-book-fiction/"
    req = urllib.request.Request(url)#,headers={'User-Agent':'Mozilla/5.0'})
    webpage = urllib.request.urlopen(req)
    soup = BeautifulSoup(webpage,"lxml")
    titles = []
    for t in soup.find_all(itemprop='name')[1:]:
        titles.append(t.text)
    
    auths = []
    for a in soup.find_all(itemprop='author'):
        auths.append(a.text[3:])
    
    
    return auths,titles



In [ ]:
bestseller()

### Part I -- Option 2

Predicting exchange rates is a difficult problem, so it totally makes sense to get help from any possible source. Look for correlations between movements of celestial objects and currency exchange rates!

 `weatherapi.com` has an astronomy api that can provide the illuminated portion of the moon for each day. It ranges from 0% (new moon) to 100% (full moon).

* Download the moon illumination for the last 100 days.
* Download all exchange rates for the last 100 days.
* Find the currency that has the highest correlation with the moon phases (illuminated portion).
* Create a plot comparing that compares the moon illumination time series to this exchange rate time series (use [this type](https://matplotlib.org/devdocs/gallery/subplots_axes_and_figures/two_scales.html) of plot).

Hint: You download the exchange rates as a dictionary; you can collect the dictionaries corresponding to each day in list; and this list can easily be converted to a dataframe, where the rows represent the days and the columns the different exchange rates.

In [ ]:
Dnow = datetime.datetime.now()
dates = [(Dnow - datetime.timedelta(days=t)).strftime("%Y-%m-%d") for t in range(50)][::-1]
dates[:3]

In [ ]:
api_key = ""
illum = []
for date in dates:
    url="http://api.weatherapi.com/v1/astronomy.json?key="+api_key+"&q=Vienna&dt="+date
    result = urllib.request.urlopen(url)
    text = result.read()
    response = json.loads(str(text,"utf-8"))
 
    illum.append(int(response['astronomy']['astro']["moon_illumination"]))


In [ ]:
plt.plot(illum)
plt.xlabel('days')
plt.ylabel('moon illumination')

In [ ]:
app_id = ""

In [ ]:
rates = []
for date in dates:
    URL = "http://openexchangerates.org/api/historical/" + date + ".json?app_id="+app_id
    result = urllib.request.urlopen(URL)
    text = result.read()
    data = json.loads(str(text,"utf-8"))
    rates.append(data['rates'])

In [ ]:
df = pd.DataFrame(rates)
df

In [ ]:
corr = df.corrwith(pd.Series(illum))
corr

In [ ]:
corr.sort_values(ascending=False)


In [ ]:
corr['FKP']

In [ ]:
maxcorr_curr =corr.idxmax()
maxcorr_curr

In [ ]:
fig, ax1 = plt.subplots()

ax2 = ax1.twinx()
ax1.plot(illum, 'g-')
ax2.plot(df[maxcorr_curr], 'b-')

ax1.set_xlabel('time')
ax1.set_ylabel('Moonphase', color='g')
ax2.set_ylabel(maxcorr_curr+' Exchange rate', color='b')

## Exercises -- NetworkX

We will sometimes use the example network from class:

In [ ]:
lesmisG = nx.read_edgelist("lesmis.csv", delimiter=',',nodetype=str, data=(("weight", int),))
lesmis_pos = nx.kamada_kawai_layout(lesmisG)

nsG = nx.read_graphml("netscience.xml")
ns_pos = {node:[data['x'],data['y']] for node,data in nsG.nodes(data=True)}
#get the largest connected component
comps = list(nx.connected_components(nsG))
cnsG = nsG.subgraph(max(comps,key=len))


### 01 Adjacency matrix

Build a directed network of `N` nodes that are labeled by intergers from `0` to `N-1` from an adjacency matrix, i.e., a two dimensional `NxN` array `A`, where `A[i][j]==1` means that there is a link pointing form `i` to `j` and `A[i][j]==0` means there is no link. 


<details><summary><u>Hint.</u></summary>
<p>

Use nested `for` loops to iterate over all possible row and column indices.
    
</p>
</details>

In [ ]:
N = 4
A = np.array([[0,1,0,0],[0,0,1,0],[0,0,0,1],[1,0,0,0]])


<details><summary><u>Solution.</u></summary>
<p>


```python
DG = nx.DiGraph()
for i in range(N):
    for j in range(N):
        if A[i][j]==1:
            DG.add_edge(i,j)
print(DG.nodes, DG.edges)
```

    
</p>
</details>


Now do the same thing using the `nx.from_numpy_array` function!

<details><summary><u>Hint.</u></summary>
<p>

Pay attention to the `create_using` attribute!
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
DG = nx.from_numpy_matrix(A, create_using=nx.DiGraph)
print(DG.nodes, DG.edges)
```

    
</p>
</details>


### 02 Erdős-Rényi model

Create a function `randnet(N,p)` that returns an undirected network with a network with `N` nodes and each node is connected with probability `p`.

<details><summary><u>Hint.</u></summary>
<p>

Iterate over all possible node pairs and add edge if `np.random.random()<p`.
                                                                             
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
def randnet(N,p):
    G =nx.Graph()
    G.add_nodes_from(range(N))
    for v in range(N):
        for w in range(v+1,N):
            if np.random.random()<p:
                G.add_edge(v,w)
    return G

randnet(10,.1).edges
```    
</p>
</details>


What you have just implemented is the Erdős-Rényi model (named after two mathematicians who were among the first to investigate it). Its definition is deceptively simple: it has a suprisingly rich and non-trivial behavior.

Of course, NetworkX has its own function to generate ER networks:

In [ ]:
nx.gnp_random_graph(10,.1)

Plot the degree distribution of one instance of the ER model with $N=1000$ nodes and $p=0.002$ connection probability.

Bonus: compare it to a Poisson distribution with mean $Np$.

<details><summary><u>Hint.</u></summary>
<p>

Collect the degrees into a list as we did for when drawing networks, and plot using `np.hist()`. Try setting the bins so that each integer is in a separate bin.
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
N=1000
p=0.002
Ger = randnet(N,p)
degs = [deg for node,deg in Ger.degree()]
plt.hist(degs,bins=np.arange(0,max(degs)+1)-.5,density=True,rwidth=.8)
plt.xlabel('degree')
plt.ylabel('probability')

#bonus
ks = np.arange(0,max(degs)+1)

#calculating the factorial upto the maximum degree
fact = np.arange(0,max(degs)+1)
fact[0]=1
fact = np.cumprod(fact)

mu = N*p
plt.plot(ks,mu**ks/fact*np.exp(-mu),'o-')
```

    
</p>
</details>


### 03 Degree distribution in real networks -- plotting  heterogeneous data

The file `dnc_email.undir_egdelist.tsv` contains a network describing an email network where the nodes are people and a link (A-B) means that there was at least one email between A and B. (The network was downloaded from [here](http://konect.cc/networks/dnc-temporalGraph/) and converted to an undirected unweighted version.)

Plot the degree distribution of the network the same way you did for the ER model, and print out the average and the maximum degree.

<details><summary><u>Hint.</u></summary>
<p>

Basically same as the previous exercise.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
dncG = nx.read_edgelist("dnc_email.undir_egdelist.tsv", delimiter='\t',nodetype=int)

degs = [deg for node,deg in dncG.degree()]
plt.hist(degs,bins=np.arange(0,max(degs)+1)-.5,density=True,rwidth=.8)
plt.xlabel('degree')
plt.ylabel('probability')
print(np.mean(degs),max(degs))
```

    
</p>
</details>


The figure is hard to read since there are many nodes with a single connection and a few nodes with many. Now try using logarithmic scale for both axes!

<details><summary><u>Hint.</u></summary>
<p>

`plt.xscale('log')` sets the x axis to log scale.    

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
degs = [deg for node,deg in dncG.degree()]
plt.hist(degs,bins=np.arange(0,max(degs)+1)-.5,density=True,rwidth=.8)
plt.xlabel('degree')
plt.ylabel('probability')
plt.xscale('log')
plt.yscale('log')
```

    
</p>
</details>


This is better, but still looks unpleasant, the bins are narrow on log scale in the tail and contain only a few events in the tail making the figure noisy. Try logarithmic binning so the bins are evenly spaced on logscale!

(Logarithmic binning means that the bins appear the same size on logscale, i.e., their width grows exponentially. If $b_0$ is the size of the initial bin, the size of $(k+1)$th bin is $b_k=a^k\cdot b_0$ where $a>0$ is a parameter.) 

Use `np.logspace()` to calculate the edges of the bins, check the documentation for details.

<details><summary><u>Hint.</u></summary>
<p>

```python
np.logspace(a,b,11)
```
creates 10 bins between $10^a$ and $10^b$.
    
Also: $10^0=1$ and $10^{\log_{10}(d_{\max})}=d_{\max}$ 
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
bins = np.logspace(0,np.log10(max(degs)),15)
plt.hist(degs,bins=bins,density=True,rwidth=.8)
plt.xlabel('degree')
plt.ylabel('probability')
plt.xscale('log')
plt.yscale('log')
```

    
</p>
</details>


This looks better! What you see is a [heavy-tailed distribution](https://en.wikipedia.org/wiki/Heavy-tailed_distribution), meaning that the distribution decays much slower than an exponential or normal distribution. There are many nodes with only a few connections, but there are few nodes with many connections, these well-connected nodes are called hubs and typically have an outsized importance with respect to processes happening on a network. Heterogeneous degree distribution is typical for complex networks, and more generally heavy-tails are common in complex systems: city sizes, wealth of individuals, and the energy of earth-quakes all follow such distributions, to name a few.

The straight line suggests that this distribution is roughly a power-law $p(k)\sim k^{-\gamma}$ (where $k$ is the degree), aka [Zipf distribution](https://en.wikipedia.org/wiki/Zipf%27s_law). However, we have to be careful with this assessment: binning averages the data which smooths out the figure, but we also loose information in the process and using it to determine the functional form of the distribution might be misleading.  When using statistical inference to fit a theoretical distribution to empirical data rely on the raw data and do not not use binning. Even using the raw data fitting the tail of a distribution is a challenging task, since by definition we only have a few observations from the tail.

Another option to visualize heterogeneous data is to plot the [complementary cumulative distribution function](https://en.wikipedia.org/wiki/Cumulative_distribution_function#Complementary_cumulative_distribution_function_(tail_distribution)): $P(k)=$ the probability that a node has degree at most $k$.

### 04 Divisiblility network

Create a directed version of the divisibility network that we did before: nodes are numbers from 1 to 100 and connect `i` to `j` with `i->j` directed link if `i` is divisible by `j`.

<details><summary><u>Hint.</u></summary>
<p>

Modify the undirected version from the other notebook.    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
N=100
DG = nx.DiGraph()
DG.add_nodes_from(range(1,N+1))
for i in range(1,N+1):
    for j in range(1,i): 
        if i % j==0: # i is always larger than j
            DG.add_edge(i,j)
```

    
</p>
</details>


Plot the network: this is a so-called *directed acyclic network*, because it does not contain cycles.

You can
* try the `shell_layout` to generate node coordinates
* generate your own coordinates: set x equal to the node label, and y a random number 

<details><summary><u>Hint.</u></summary>
<p>

If generating our own coordinates the position of `i` should be [i,np.random.random()].
    
The positions of the nodes have to be passed as a dictionary to the draw function, where the node labels are keys and their positions are the values.
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
plt.figure(figsize=(12,12))
# layout
pos = nx.shell_layout(DG)
pos = {i:[i,np.random.random()] for i in range(1,N+1)}
# draw nodes
nx.draw_networkx_nodes(DG, pos,node_size=400,edgecolors='w')
nx.draw_networkx_edges(DG, pos, width=1,edge_color="#99999999")
# draw labels
nx.draw_networkx_labels(DG, pos, font_size=15)

plt.axis('off');
```

    
</p>
</details>


In a directed network you can calculate the in-degree (number of links pointing at the node) and out-degree (number of nodes pointing out of the node). Using NetworkX you calculate the out-degree of all nodes with `DG.out_degree()`.

How many prime numbers are there in the network? (One is not considered a prime.)

<details><summary><u>Hint.</u></summary>
<p>

Prime numbers are only divisible by 1. 
</p>
</details>



<details><summary><u>Solution.</u></summary>
<p>


```python
print("The number of prime numbers between 1 and 100 =",[indeg for i, indeg in DG.out_degree()].count(1))
```

    
</p>
</details>


You can find the longest path in a directed acyclic network using the `nx.dag_longest_path` function. Which number has the most prime factors? E.g., 50 has 3 prime factors since $50=5\cdot5\cdot2$

<details><summary><u>Hint.</u></summary>
<p>

Traversing an edge is a division, starting from `i` and reaching `1` means that we factorized `i`.
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
longest_path = nx.dag_longest_path(DG)
print(longest_path)
print(f"{longest_path[0]} has to most prime factors, all together {len(longest_path)-1}")
```

    
</p>
</details>


## Exercises -- scipy-stats

### 05 Normality test

You have a random samples `X` and `Y`, determine whether they are normally distributed or not. Consult the documentation of `scipy.stats`.

<details><summary><u>Hint.</u></summary>
<p>

One possibility is to use this [test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.normaltest.html). 
    
</p>
</details>


In [ ]:
X = np.array([ 1.09087983,  9.03450954,  7.7792193 ,  5.54347729,  0.5589584 ,
        3.07076011,  3.12572162,  4.92747005,  4.96121311,  5.68349957,
        3.3469864 ,  4.56833257,  2.48169444,  3.51976994,  4.17342725,
        4.15711623,  4.38249048,  3.87198581,  3.45200412,  4.33304951])
Y = np.array([0.47502293, 0.09568451, 0.30165286, 0.70134313, 0.74012117,
       0.13758756, 0.36993677, 1.63704143, 1.91214249, 2.36786318,
       1.65867764, 0.3198418 , 2.23937978, 3.69685446, 5.57572031,
       1.68093789, 0.78964349, 0.54619459, 0.23462287, 6.93310479])

<details><summary><u>Solution.</u></summary>
<p>


```python
print(stats.normaltest(X))
print(stats.normaltest(Y))
```

    
</p>
</details>


### 06 Cheeseburgers and fries

Vincent Vega and Jules Winnfield are talking about fastfood restaurants again. Vincent claims that he visited over 100 restaurants in town and there is a clear negative correlation between the quality of fries and cheeseburgers, while Jules says this is impossible, it only makes common sense that places that have good fries also have good burgers.

As the two argue, they get increasingly agitated. Help these two good friends figure out who is right!

Load the file `fastfoodrating.csv` which contains the ratings of all burger joints in town, it has a separate rating for cheesburgers and for fries. It also has a column indicating if Vincent has visited it or not. Use linear regression to check both the statement of Vincent and the statement of Jules.

<details><summary><u>Hint.</u></summary>
<p>

Load the data into a pandas dataframe and do a linear regression on all the restaurant and after subsetting for the restaurants with `Visited` column `True`.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
df = pd.read_csv('fastfoodrating.csv')
print(stats.linregress(df.Cheeseburger,df.FrenchFries))
print(stats.linregress(df.Cheeseburger[df.Visited],df.FrenchFries[df.Visited]))
```

    
</p>
</details>


The friendship is saved! But how can this be? Create a plot that explains how both statements can be true at the same time!

<details><summary><u>Hint.</u></summary>
<p>

Check out [Berkson's paradox](https://www.google.com/search?q=Berkson%27s+paradox&sxsrf=ALiCzsYza4WX8C1AkRQ0rIbvElXovI8I_Q:1668582735764&source=lnms&tbm=isch&sa=X&ved=2ahUKEwiK__iJk7L7AhUfg_0HHbSnCXwQ_AUoAXoECAEQAw&biw=1363&bih=682&dpr=2.73)

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
ax=df.plot(kind='scatter',x='Cheeseburger',y='FrenchFries')
df[df.Visited].plot(kind='scatter',x='Cheeseburger',y='FrenchFries',ax=ax,color='r')
```

    
</p>
</details>


### 07 Simpson's paradox --  Discussion exercise

Wikipedia [says](https://en.wikipedia.org/wiki/Simpson%27s_paradox): Simpson's paradox is a phenomenon in probability and statistics in which a trend appears in several groups of data but disappears or reverses when the groups are combined. And provides the following figure to illustrate the paradox [here](https://upload.wikimedia.org/wikipedia/commons/f/fb/Simpsons_paradox_-_animation.gif).

Recreate a non-animated version of this figure with two groups and two subplots showing the relation between x and y without taking the grouping into account and with grouping.

Generate the data! There are many different ways to generate data that exhibits Simpson's paradox, select a method that is easy for you to implement.

<details><summary><u>Hint.</u></summary>
<p>

One possible way to generate linearly correlated x-y pairs is to
* Generate a range of x values, for example using `np.linspace`, np.random.uniform` or `np.arange`.
* Generate y values as a linear function of x, for example, `a*x+b`
* Add random noise to y and/or x.

Experiment with the parameters (range of x values, `a` and `b`, noise level) so you can reproduce a Simpson's paradox. Recreate the animated figure without the animation using two subplots.
    
</p>
</details>

**Share your solution on Slack. We will discuss the problem during class together.**

## Exercises -- scikit-learn, Predicting the NBA playoffs

This last portion of the notebook walks you through a longer example for scikit-learn building a logistic regression model to predict the outcome of basketball games. Along the way we will use tools we have covered over the course.

An [NBA](https://en.wikipedia.org/wiki/National_Basketball_Association) is an American professional basketball league. Each season has to parts: The [regular season](https://en.wikipedia.org/wiki/National_Basketball_Association#Regular_season), where each team plays the same number of games and the teams are ranked based on their number of wins. Following the regular season, the top ranked teams advance to the [playoffs](https://en.wikipedia.org/wiki/National_Basketball_Association#Playoffs), the playoffs follow a tournament format.

We will download and clean the results of the 2017-2018 NBA season and we will attempt to predict the playoffs.

### 08 Pandas -- collect and clean the data

This [website](https://www.basketball-reference.com/leagues/NBA_2018_games-october.html) contains NBA match results and statistics.

#### 08a Download the data

Write code that downloads all the results from the 2017-2018 season using `pd.read_html()` and collects them in a single pandas dataframe `season_df`.

<details><summary><u>Hint.</u></summary>
<p>
    
* The data is stored on a separate page for each month, use a `for` loop to iterate over the months.
* Use `pd.read_html()` to download the table. This function returns a list of dataframes each corresponding to a table on the website, this website happens to have only one table.
* Collect the dataframes for each month in a list and stitch them together using `pd.concat()`.
    
</p>
</details>

In [ ]:
months = ['october', 'november', 'december', 'january', 'february', 'march', 'april','may','june']


<details><summary><u>Solution.</u></summary>
<p>
    
```python
monthly_dfs = []
for month in months:
    df = pd.read_html("https://www.basketball-reference.com/leagues/NBA_2018_games-"+month+".html")[0]
    
    monthly_dfs.append(df)

season_df = pd.concat(monthly_dfs, axis=0)
```
    
</p>
</details>

#### 08b Drop and rename columns

We will only use the columns `'Date', 'Visitor/Neutral', 'PTS', 'Home/Neutral', 'PTS.1'`, drop every other column. Rename the columns `'date', 'vis', 'vis_pts','home', 'home_pts'` to make their meaning more clear and to reduce the amount of typing need.

<details><summary><u>Hint.</u></summary>
<p>
  
You can select a subset of columns and overwrite the dataframe with `df=df[[col1,col2,...]]` or you can use the `df.drop()` method.
    
Columns can be renamed by overwriting `season_df.columns=[new_col_name1,new_col_name2,...]`.  
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
season_df = season_df[['Date','Visitor/Neutral','PTS','Home/Neutral', 'PTS.1']]
season_df.columns = ['date','vis','vis_pts','home','home_pts']
season_df
```
    
</p>
</details>

#### 08c Drop unwanted rows

If you check out the [website](https://www.basketball-reference.com/leagues/NBA_2018_games-october.html), you can see that the header is repeated a number of times. Furthermore there is a row in the table in April indicating the start of the playoffs. Remove these lines from the dataframe!

<details><summary><u>Hint.</u></summary>
<p>

The rows you need to drop contain the header instead of data. For example, the `date` column usually contains a string representing a date, but in the rows to be dropped it contains the word `'Date'` or `'Playoffs`'.
    
</p>
</details>

<details><summary><u>Hint 2.</u></summary>
<p>

One way to drop the rows is to
* Create a mask that is `True` for rows that you would like to keep, i.e., the date column is not equal to 'Date'` and `'Playoffs`'
* The symbol for the elementwise logical and is `&`, e.g., `mask = mask1 & mask2` returns a mask where `mask[i]` is `True` if both `mask1[i]` and `mask2[i]` are `True`
* Select the rows you would like to keep using this mask, e.g., `df[mask]`.
</p>
</details>

Note that if you do an operation like `df=df[mask]`, you might get warning messages if you modify `df` (for example you add a new column). This is because `df[mask]` will be a view and not a copy of the original `df` if there are variables still referencing `df`. To make sure you create a new copy instead of a view, use `df=df[mask].copy()`.

In [ ]:
mask = (season_df.date!='Playoffs') & (season_df.date!='Date')
season_df = season_df[mask].copy()

<details><summary><u>Solution.</u></summary>
<p>
    
```python
mask = (season_df.date!='Playoffs') & (season_df.date!='Date')
season_df = season_df[mask].copy()
```
    
</p>
</details>

#### 08d Convert columns

All data is stored as strings, we need to convert them into a useful format. Convert the `'date'` colunm to a datetime object and convert the columns containing the points to integers. 

<details><summary><u>Hint.</u></summary>
<p>
    
Use the function `pd.to_datetime()`. To specify the date format check out the meaning of `%Y`, `%b`, `%d`, and `%a`.
    
To convert an entire column to integer, you can use `df['col'].astype(np.int)`. (Or `pd.to_numeric()` which has more options to handle errors in conversions.)
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
season_df['date'] = pd.to_datetime(season_df.date, format="%a, %b %d, %Y")

season_df['vis_pts']=season_df['vis_pts'].astype(np.int)
season_df['home_pts']=season_df['home_pts'].astype(np.int)

season_df.reset_index(inplace=True, drop=True)
season_df
```
    
</p>
</details>

### 09 Datetime -- Regular season mask

We are planning to train a model using the games in the regular season to predict the playoffs, so we have to separate the data. In the 2017-2018 season the playoffs started on April 14. Create two masks `reg_mask` and `playoff_mask` such that applied to `season_df` select the regular season and the playoff, respectively. 

<details><summary><u>Hint.</u></summary>
<p>
    
Create a datetime object for April 14, then you can use the coperison operators `<`, `<=`, etc. to compare dates.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
po=datetime.datetime(2018,4,14)

reg_mask = season_df.date<po
playoff_mask = season_df.date>=po

season_df[playoff_mask]
```
    
</p>
</details>

### 10 sklearn, numpy, sorting -- Logistic model/Bradley-Terry model for paired comparisons

A common way to model sports matches (or other paired comparisons) is to use the [Bradley-Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model). In this model we fit a parameter $w_i$ for each team $i$ indicating their strength, and the probability of $i$ winning a against $j$ is given by the logistic function
$$\frac{e^{w_i}}{e^{w_i}+e^{w_j}}=\frac{1}{1+e^{-(w_i-w_j)}} $$
It turns out that fitting this model is a special case of logistic regression (see wikipedia for details). For each game you have to create a feature vector `x` that has length equal to the number of teams in the league:
* `x[i]=1` indicates that `i` is the home team,
* `x[i]=-1` indicates that `i` is the visitor team,
* `x[i]=0` means that `i` is not playing.
For example, if there are 4 teams and team 0 is hosting team 2, the feature vector is `x=[1,0,-1,0]`.
The outcome that you we predict is `1` if the home team wins and `0` if the visitor.

In this exercise we will prepare arrays containing the feature vectors and the outcomes and we will train a logistic model (aka a Bradley-Terry model) to predict the playoff.

#### 10a Team name to index

sklearn works with numpy arrays and arrays cannot be indexed by team names. Let's create a mapping between team names and integers, i.e., `team0`->`0`.

Create a dictionary that has team names as keys and the corresponding integers as values.

<details><summary><u>Hint.</u></summary>
<p>
  
* We already created a list `teams` containing the unique team names when in the networkx exercise.
* Create a dictionary `teams_map` where `teams_map[team_name]=(the position of teams_name in the list teams)`
* You can for example use `enumerate()`.

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
teams_map  = {team:i for i,team in enumerate(teams)}
```
    
</p>
</details>

#### 10b Prepare data for sklearn

Create a 2 dimensional numpy array `features` full of zeros that has 
* the same number of colunms as teams
* the same number of rows as matches in the season.
For each row `i` of the array set the column `features[i][teams_map[home_team]]` to 1 and `features[i][teams_map[visitor_team]]` to -1.

Create a 1 dimensional numpy array `outcomes` such that `outcomes[i]=1` if the home team won and 0 otherwise

<details><summary><u>Hint.</u></summary>
<p>

Create arrays initially filled with zeros using `np.zeros(size)`
    
You can fill these arrays up with a `for` loop iterating over `season_df.iterrows()`.
    
Or if you would like to avoid explicit `for` loops, you can write a function that takes a row as input and modifies arrays `features` and `outcomes` (and does not return anything). Then  you use `df_season.apply()` to apply the function to each row.

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
features = np.zeros((len(season_df),len(teams)))
outcomes = np.zeros(len(season_df))

#solution 1
for i,row in season_df.iterrows():
    features[i,teams_map[row.home]]= 1
    features[i,teams_map[row.vis]] =-1
    outcomes[i] = 1 if row.home_pts>row.vis_pts else 0
    
#solution 2
def f(row,features,outcomes):
    features[row.name,teams_map[row.home]]= 1
    features[row.name,teams_map[row.vis]] =-1
    
    outcomes[row.name] = 1 if row.home_pts>row.vis_pts else 0
    return

season_df.apply(lambda row: f(row,features,outcomes), axis=1)
```
    
</p>
</details>

#### 10c Fit logistic model

At this point we usually randomly split the data into a training and a test set. In our case, however, we would like to train the data based on the regular season and test the predicitions on the playoff. Luckily we already have a way to split the data this way: use the previously created `reg_mask` and `playoff_mask`!

Then do the logistic regression using sklearn. Use `LogisticRegression(fit_intercept=False)` to make sure we fit the Bradley-Terry model. Plot the confusion matrix!

<details><summary><u>Hint.</u></summary>
<p>
    
To get the training set simply use `features[reg_mask]` and `outcomes[reg_mask]`. Otherwise the process is very similar to what we did in the other notebook.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
logistic = linear_model.LogisticRegression(fit_intercept=False)
logistic.fit(features[reg_mask],outcomes[reg_mask]);

predictions = logistic.predict(features[playoff_mask])
cm = metrics.confusion_matrix(outcomes[playoff_mask],predictions)

plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt="d", linewidths=.5, square = True, cmap = 'Blues_r');
plt.ylabel('Actual label');
plt.xlabel('Predicted label');
plt.title(f'Accuracy Score: {logistic.score(features[playoff_mask], outcomes[playoff_mask]):.3f}', size = 12);
```
    
</p>
</details>

In the confusion matrix, each predicted game is labelled by their most likely outcome, independent of it has 51% chance or 99%. It is also possible to obtain the probabilities using `LogisticRegression.predict_proba`.

#### 10d Top teams

The coefficients `w_i` indicate how strong a team is, you can access the values of the best fit coefficients using `logistic.coef_`. Print out the name of the top 10 teams and their corresponding `w_i` values.

<details><summary><u>Hint.</u></summary>
<p>
    
This is almost the same as we did for the page rank. The $w_i$ of team `i` is `logistic.coef_[0][teams_map[i]]`.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
sorted_teams = sorted(teams, key= lambda t: -logistic.coef_[0][teams_map[t]])
for rank,t in enumerate(sorted_teams[:10]):
    print(rank+1,t,logistic.coef_[0][teams_map[t]], sep='\t')
```
    
</p>
</details>

These values can be interpreted easily. For example, our model predicts that the first team wins against the second with probability:

In [ ]:
w_diff = logistic.coef_[0][teams_map[sorted_teams[0]]]-logistic.coef_[0][teams_map[sorted_teams[1]]]
1/(1+np.exp(-w_diff))

And the 9th against the 10th:

In [ ]:
w_diff = logistic.coef_[0][teams_map[sorted_teams[8]]]-logistic.coef_[0][teams_map[sorted_teams[9]]]
1/(1+np.exp(-w_diff))

Quite ambiguous prediction!

In principle, it is also possible to assign confidence intervals to the `w_i` coefficients, however, in machine learning the focus is on out-of-sample prediction and not p-values, model validation is done using the test data that we put aside before the model training. Therefore sklearn cannot be used to calculate confidence intervals, and you would have to use statistics modules such as statsmodels for this.

### 11 matplotlib, scipy.stats, seaborn -- Home team advantage

Can we improve this prediction? It is commonly thought that playing at your home stadium increases your chances of winning. Let's investigate if this is true.

#### 11a Histogram

Create a plot showing histograms of the points scored by the home team and the points scored by the visitor team. Add two vertical lines indicating the average for the home and visitor teams.

Select the number of bins, color, transparency, etc. so that the plot looks ok.

<details><summary><u>Hint.</u></summary>
<p>
    
Use plt.hist(). You can set the transparency of the histograms using the attribute `alpha=x` where `x` is between 0 and 1.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
plt.hist(season_df['vis_pts'],bins=15,alpha=1.,rwidth=.9,color='#aa00aa',label='Visitor')
plt.hist(season_df['home_pts'],bins=15,alpha=.8,rwidth=.9,color='#00aa00',label='Home')
plt.axvline(season_df['vis_pts'].mean(),color='k',linewidth=2,label="Visitor average")
plt.axvline(season_df['home_pts'].mean(),color='k',linewidth=2,linestyle='dashed',label="Home average")
plt.xlabel('Points')
plt.ylabel('Count')
plt.legend()
```
    
</p>
</details>

#### 11b Linear regression with scipy.stats

The home teams on average seem to score more than the visitors. Is this difference statistically significant? It is unlikely that the home team points and the visitor points at game are independent from each other. Therefore, let's first let's check the their relationship. Use scipy.stats and linear regression the find the association between `season_df['vis_pts']` and `season_df['home_pts']`. 

<details><summary><u>Hint.</u></summary>
<p>
    
This is the same as in the other notebook.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
result_yearly = stats.linregress(season_df['vis_pts'], season_df['home_pts'])
result_yearly
```
    
</p>
</details>

#### 11c Linear regression with seaborn

There is a positive relationship between the scores. We can use seaborn to preform the linear regression and plot it in one step. Use `sns.jointplot` to create a regression plot.

<details><summary><u>Hint.</u></summary>
<p>

Set the attribute `kind="reg"` for the regression plot. You can change the color of the regression line without affecting the rest of the colors using `joint_kws={'line_kws':{'color':'#aaaa00'}`.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
g = sns.jointplot(
    data=season_df,
    x="vis_pts", y="home_pts",
    kind="reg", joint_kws={'line_kws':{'color':'#aaaa00'}}
)
```
    
</p>
</details>

#### 11d t-test

Let's test if the home and away score difference is statistically significant. In the other notebook we used `scipy.stats.ttest_ind()` to preform a t-test on two samples; however, that test assumes that the two samples are independent, and we know that is not true for `season_df['vis_pts']` and `season_df['home_pts']`. So instead we have to use the paired version `stats.ttest_rel()`.

Preform the test and print out the statistic and the p-value.

<details><summary><u>Hint.</u></summary>
<p>
This also almost identical to the t-test in the other notebook.
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
statistic, pvalue = stats.ttest_rel(season_df['vis_pts'], season_df['home_pts'])
print(statistic, pvalue)
```
    
</p>
</details>

We found that there is indeed a significant home team advantage. We can incorporate this into the logistic model by introducing a bias parameter to the Bradley-Terry model as:
$$\frac{1}{1+e^{-(w_i-w_j + b)}}, $$
where $b$ does not depend on the teams and always favors the home team.

In terms of the logistic model this simply means that we fit the intercept. You can simply do this by going back to exercise 03 and modify `fit_intercept` to `True`:
```python
logistic = linear_model.LogisticRegression(fit_intercept=True)
```

### 12 BONUS: BeautifulSoup, JSON -- Travel distance

[Travelling](https://en.wikipedia.org/wiki/Traveling_(basketball)) in basketball is considered a bad thing.

One might hypothesise the home team advantage is greater if the visitor team has to travel far. Let's check this out. First we have to obtain the location of the home court of each team and then incorporate this into the model.


#### 12a Get coordinates of arena

Write a function `ArenaCoords(team)` that takes a team name as input and uses the internet to return the coordinates of their home court. 

Use this function to create a dictionary that has the team names as keys and tuples representing the coordinates as values.

<details><summary><u>Hint.</u></summary>
<p>
    
* Scrape the wikipedia page of the team to get the name of their arena.
* Use the api of [OpenStreetMap](https://nominatim.org/release-docs/latest/api/Search/) to get the coordinates of the arena based on its name.
    
</p>
</details>


<details><summary><u>Detailed hint.</u></summary>
<p>

The steps are long to describe, but the code in the end is just a few lines, most of which can be copy-pasted from previous notebooks.
    
**Wikipedia**:
* The name of the wikipedia page is the team name such that the `" "` spaces are replaced by `"_"` underscores. For example: https://en.wikipedia.org/wiki/Golden_State_Warriors
* On every team's page the name of their arena is stored in a table, search for the table row using the `soup.find(condition(tag))` which returns the first match for the `condtion`.
* Define the `condition(tag)` function such that:
    * It looks for a `<tr>` table row html tag
    * It contains a `<th>` table header tag inside it (`tag.th!=None`)
    * The string contained inside the `<th>` is equal to "'Arena'"
* The table row you found contains a `<td>` table data tag which contains the name of the arena 
    
    
**Web api**:
* The nominatim web api works (without registration) by requesting urls of the format:
https://nominatim.openstreetmap.org/search?q=Budapest&format=json    
where Budapest can be replaced with any location. The api finds a number of places matching the location string you give it and returns a json object with a bunch of info. Open the above url in your browser to check out a result.

* Construct an url where the location is replaced by the name of the arena. Watch out! The location cannot contain spaces you have to replace it with the escape sequence "%20", it also cannot contain `"&"` which has to be replaced by "%26".
* The json object you get is a list of dictionaries, each dictionary corresponding to place that nominatim thinks is a match. The places have a `"type"`, the arenas have type equal to 'stadium', 'arena', or 'attraction'. Pick the first entry that matches this type.
* Convert the coordinates from stings to floats
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
def ArenaCoords(team):
    webpage = urllib.request.urlopen("https://en.wikipedia.org/wiki/"+team.replace(" ","_"))
    soup = BeautifulSoup(webpage,"lxml")
    loc_row = soup.find(lambda tag: tag.name=='tr' and tag.th!=None and tag.th.string=="Arena")
    #print(team,loc_row.td.text)
    
    location = loc_row.td.text.replace(" ","%20").replace("&","%26")
    osm_url = "https://nominatim.openstreetmap.org/search?q="+location+"&format=json"
    #print(osm_url)
    result = urllib.request.urlopen(osm_url)
    text = result.read()
    json_coords = json.loads(text)
        
    for place in json_coords:
        if place['type']=='stadium' or place['type']=='arena' or place['type']=='attraction':
            return float(place['lat']),float(place['lon'])
     
    return None,None

coords = {team:ArenaCoords(team) for team in teams}
```
    
</p>
</details>

#### 12b Travel distance

Use our old favorite [Haversine formula](https://en.wikipedia.org/wiki/Haversine_formula) to calculate the distance the visitors have to travel. Add it to the `season_df` as new column called `"travel_dist"`.

<details><summary><u>Hint.</u></summary>
<p>
    
Use `df.apply()` together with the dictionary you created in the previous exercise to apply the Haversine formula to each row.
    
</p>
</details>

In [ ]:
#Haversine formula
def latlongdist(lat1,long1,lat2,long2):
    rlat1 = math.radians(lat1)
    rlat2 = math.radians(lat2)
    rlong1 = math.radians(long1)
    rlong2 = math.radians(long2)
    dlat = rlat2 - rlat1
    dlong = rlong2 - rlong1
    a = math.sin(dlat / 2)**2 + math.cos(rlat1) * math.cos(rlat2) * math.sin(dlong / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return 6371.0 * c

<details><summary><u>Solution.</u></summary>
<p>
    
```python
season_df['travel_dist'] = season_df.apply(lambda row: latlongdist(
                                                            coords[row.home][0],
                                                            coords[row.home][1],
                                                            coords[row.vis][0],
                                                            coords[row.vis][1])
                                          ,axis=1)
```
    
</p>
</details>

#### 12c Extend feature vector and predict

Create a new array `features2` thah has one more column than `features`, which contains the travel distance.

It typically helps the numerical stability of training if the entries in the feature vector have the same magnitude. Divide the new column by the average travel distance.

<details><summary><u>Hint.</u></summary>
<p>
    
* Create a new array with shape `(features.shape[0],features.shape[1]+1)`
* Use slicing to select a subarray excluding the last column and set the slice to be equal to the old array `features`
* Use slicing to select the last column and set the slice to be equal to the `travel_dist` column of the dataframe.
* Divide the last column by its mean.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
features2 = np.zeros((features.shape[0],features.shape[1]+1))
features2[:,:-1]=features
features2[:,-1]= season_df['travel_dist'].values
features2[:,-1]/=features2[:,-1].mean()
features2[1]
```
    
</p>
</details>

#### 12d Train the model

Fit the logistic model again, but instead of including an intercept, use the new extended feature vector.

Plot the confusion matrix.

<details><summary><u>Hint.</u></summary>
<p>
    
Almost identical to the first logistic regression in the notebook. Make sure that `fit_intercept=False`.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
logistic_trav = linear_model.LogisticRegression(fit_intercept=False)
logistic_trav.fit(features2[reg_mask],outcomes[reg_mask]);

predictions_trav = logistic_trav.predict(features2[playoff_mask])
cm_trav = metrics.confusion_matrix(outcomes[playoff_mask],predictions_trav)

plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt="d", linewidths=.5, square = True, cmap = 'Blues_r');
plt.ylabel('Actual label');
plt.xlabel('Predicted label');
score = logistic_trav.score(features2[playoff_mask], outcomes[playoff_mask])
plt.title(f'Accuracy Score: {score:.3f}', size = 12);
```
    
</p>
</details>

Sadly, this didn't help.

This seems like the right place to end the course.

## Final problem

Use the `dolphins.csv` or download a network from http://konect.cc and create a visualization:
* Node size should be proportional to degree
* Color the nodes based on PageRank, bewteenness or try something [new](https://networkx.org/documentation/stable/reference/algorithms/centrality.html).
* Use matplotlib to set the title of the plot to be the name of your dataset

`dolphins.csv` contains a network of 62 dolphins, where links mean that they often hang out together. Pay attention: although the file is a `csv` the columns are separated by tabs (`\t`).

## Extra credit challenge

You have bunch of empty boxes. To store them efficiently you are planning to place them inside each other. Each box has three dimensions `(h, w, l)`, assume that you can place box A inside box B if all of its dimensions are smaller, i.e., `hA<hB`, `wA<wB`, and `lA<lB`. (You don't like to rotate your boxes!) Write a function that
* Takes a list of tuples as input, where each tuple describes the dimensions of a box. No two boxes have exactly the same size.
* Returns the longest sequence of boxes that you can place inside of each other in a nested fashion.

Example

Input:
```
[(10,10,5),(1,1,1),(2,2,8),(9,5,3)]
```
Output:
```
[(10,10,5),(9,5,3),(1,1,1)]
```